In [68]:
import timm
import torch
import torch.nn as nn
import os
import cv2
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold
import timm
from tqdm import tqdm
import glob
import random
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import string

In [69]:
CSV_PATH = "dataset_orientation/labels.csv"
IMAGES_DIR = "dataset_orientation/images"
TEST_DIR = "test/images"
MODEL_PATH = "best_model.pth"
BATCH_SIZE = 64
EPOCHS = 5
LR = 5e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
TARGET_H = 48
TARGET_W = 192

In [77]:
model = timm.create_model(
    "lcnet_100",
    pretrained=True,
    num_classes=1,
    drop_rate=0.2
)

In [71]:
def resize_with_pad(img, target_h=TARGET_H, max_w=TARGET_W):
    h, w, _ = img.shape
    scale = target_h / h
    new_w = min(int(w * scale), max_w)
    
    resized = cv2.resize(img, (new_w, target_h), interpolation=cv2.INTER_LINEAR)
    
    canvas = np.full((target_h, max_w, 3), 128, dtype=np.uint8)
    canvas[:, :new_w] = resized
    
    tensor = canvas.astype(np.float32) / 255.0
    tensor = (tensor - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]
    tensor = np.transpose(tensor, (2, 0, 1))
    return torch.tensor(tensor, dtype=torch.float32)

In [72]:
def apply_realistic_artifacts(img):
    """
    Симуляция артефактов реальных кропов:
    блюр, JPEG-артефакты, срез краев, шум соседних строк, игра контраста.
    """
    h, w = img.shape[:2]

    # 1. Мусор от соседних строк / обрезка границ детектора (в 40% случаев)
    if random.random() < 0.4:
        strip_h = random.randint(2, max(3, int(h * 0.15)))
        color = tuple(int(c) for c in random.choice([img[0, 0], [128, 128, 128], [255, 255, 255], [0, 0, 0]]))
        if random.random() < 0.5:
            # Черточка или шум сверху
            cv2.line(img, (0, strip_h // 2), (w, strip_h // 2), color, random.randint(1, 3))
        else:
            # Снизу
            cv2.line(img, (0, h - strip_h // 2), (w, h - strip_h // 2), color, random.randint(1, 3))

    # 2. Размытие / Потеря резкости (в 50% случаев)
    if random.random() < 0.5:
        k = random.choice([3, 5])
        if random.random() < 0.6:
            img = cv2.GaussianBlur(img, (k, k), 0)
        else:
            # Моушн-блюр по горизонтали
            kernel = np.zeros((k, k))
            kernel[k // 2, :] = 1.0 / k
            img = cv2.filter2D(img, -1, kernel)

    # 3. Агрессивное сжатие JPEG (в 40% случаев)
    if random.random() < 0.4:
        quality = random.randint(15, 60)
        _, enc = cv2.imencode(".jpg", img, [int(cv2.IMWRITE_JPEG_QUALITY), quality])
        img = cv2.imdecode(enc, cv2.IMREAD_COLOR)

    # 4. Падение контраста (выцветший / тисненый текст в 35% случаев)
    if random.random() < 0.35:
        alpha = random.uniform(0.3, 0.7)  # Контраст
        beta = random.randint(-40, 40)    # Яркость
        img = np.clip(alpha * img + beta, 0, 255).astype(np.uint8)

    # 5. Цифровой шум сенсора (в 30% случаев)
    if random.random() < 0.3:
        noise = np.random.normal(0, random.uniform(5, 20), img.shape).astype(np.float32)
        img = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)

    return img

In [73]:
def resize_with_pad(img, is_train=False, target_h=TARGET_H, max_w=TARGET_W):
    if is_train:
        img = apply_realistic_artifacts(img)

    h, w, _ = img.shape
    scale = target_h / max(h, 1)
    new_w = min(int(w * scale), max_w)
    
    resized = cv2.resize(img, (max(new_w, 1), target_h), interpolation=cv2.INTER_LINEAR)
    
    # Паддинг серым цветом (128)
    canvas = np.full((target_h, max_w, 3), 128, dtype=np.uint8)
    canvas[:, :new_w] = resized
    
    # Нормализация
    tensor = canvas.astype(np.float32) / 255.0
    tensor = (tensor - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]
    tensor = np.transpose(tensor, (2, 0, 1))
    return torch.tensor(tensor, dtype=torch.float32)

In [ ]:
class CropDataset(Dataset):
    def __init__(self, df, img_dir, is_train=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row["filename"])
        img = cv2.imread(img_path)
        
        if img is None:
            img = np.zeros((TARGET_H, TARGET_W, 3), dtype=np.uint8)

        tensor = resize_with_pad(img, is_train=self.is_train)
        
        if self.is_train:
            target_val = 0.95 if row["label"] == 180 else 0.05
        else:
            target_val = 1.0 if row["label"] == 180 else 0.0
            
        label = torch.tensor(target_val, dtype=torch.float32)
        return tensor, label

In [75]:
def compute_brier_metric(probs, targets):
    brier_score = np.mean((probs - targets) ** 2)
    return 1.0 - brier_score

In [ ]:
print(f"Используемое устройство: {DEVICE}")
df = pd.read_csv(CSV_PATH)

gkf = GroupKFold(n_splits=5)
train_idx, val_idx = next(gkf.split(df, groups=df["group_id"]))

train_df = df.iloc[train_idx]
val_df = df.iloc[val_idx]
print(f"Выборки: Train = {len(train_df)} сэмплов, Val = {len(val_df)} сэмплов")

train_loader = DataLoader(CropDataset(train_df, IMAGES_DIR), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(CropDataset(val_df, IMAGES_DIR), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

model.to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_score = 0.0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for inputs, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        inputs = inputs.to(DEVICE)
        targets = targets.to(DEVICE).unsqueeze(1)
        
        optimizer.zero_grad()
        logits = model(inputs)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * len(targets)

    train_loss /= len(train_df)
    scheduler.step()

    model.eval()
    val_probs, val_targets = [], []
    with torch.no_grad():
        for inputs, targets in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
            inputs = inputs.to(DEVICE)
            logits = model(inputs)
            probs = torch.sigmoid(logits).squeeze(1).cpu().numpy()
            
            val_probs.extend(probs)
            val_targets.extend(targets.numpy())

    val_probs = np.array(val_probs)
    val_targets = np.array(val_targets)
    score = compute_brier_metric(val_probs, val_targets)

    print(f"Epoch {epoch+1}: Train Loss = {train_loss:.4f} | 1 - Brier Score = {score:.5f}")

    if score > best_score:
        best_score = score
        torch.save(model.state_dict(), "best_model_3.pth")
        print(f"-> Сохранен лучший чекпоинт с метрикой {best_score:.5f}")

In [79]:
def preprocess_image(img, target_h=TARGET_H, max_w=TARGET_W):
    h, w, _ = img.shape
    scale = target_h / max(h, 1)
    new_w = min(int(w * scale), max_w)
    
    resized = cv2.resize(img, (max(new_w, 1), target_h), interpolation=cv2.INTER_LINEAR)
    
    canvas = np.full((target_h, max_w, 3), 128, dtype=np.uint8)
    canvas[:, :new_w] = resized
    
    tensor = canvas.astype(np.float32) / 255.0
    tensor = (tensor - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]
    tensor = np.transpose(tensor, (2, 0, 1))
    return torch.tensor(tensor, dtype=torch.float32)

In [80]:
class TestDataset(Dataset):
    def __init__(self, image_paths):
        self.image_paths = image_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        filename = os.path.basename(path)
        img = cv2.imread(path)
        
        if img is None:
            img = np.zeros((TARGET_H, TARGET_W, 3), dtype=np.uint8)

        orig_tensor = preprocess_image(img)
        rot_img = cv2.rotate(img, cv2.ROTATE_180)
        rot_tensor = preprocess_image(rot_img)

        return filename, orig_tensor, rot_tensor

In [ ]:
test_extensions = ("*.jpg", "*.jpeg", "*.png")
image_paths = []
for ext in test_extensions:
    image_paths.extend(glob.glob(os.path.join(TEST_DIR, ext)))
    image_paths.extend(glob.glob(os.path.join(TEST_DIR, ext.upper())))

image_paths = sorted(list(set(image_paths)))
print(f"Найдено тестовых картинок: {len(image_paths)}")
print(f"Устройство для инференса: {DEVICE}")

In [ ]:
model = timm.create_model('lcnet_100', pretrained=False, num_classes=1)
model.load_state_dict(torch.load("best_model_3.pth", map_location=DEVICE))
model.to(DEVICE)
model.eval()

test_loader = DataLoader(
    TestDataset(image_paths),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

filenames_list = []
probs_list = []

with torch.no_grad():
    for filenames, orig_batch, rot_batch in tqdm(test_loader, desc="Инференс теста c TTA"):
        orig_batch = orig_batch.to(DEVICE)
        rot_batch = rot_batch.to(DEVICE)
        
        # TTA: усреднение прямого и инвертированного прогона
        p1 = torch.sigmoid(model(orig_batch)).squeeze(1).cpu().numpy()
        p2 = 1.0 - torch.sigmoid(model(rot_batch)).squeeze(1).cpu().numpy()
        final_probs = (p1 + p2) / 2.0
        
        filenames_list.extend(filenames)
        probs_list.extend(final_probs)

clean_ids = [os.path.splitext(f)[0] for f in filenames_list]

safe_probs = np.clip(probs_list, 0.03, 0.97)

sub_df = pd.DataFrame({
    "image_id": clean_ids,
    "p_180": safe_probs
})

sub_df.to_csv("submission.csv", index=False)

print(f"\nФайл сохранен в submission.csv")
print(sub_df.head(10))